In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_classif

In [ ]:
df = pd.read_csv('data/train.csv')
df

,matchId,playerId,playType,bodyPart,x,y,interveningOpponents,interveningTeammates,interferenceOnShooter,minute,second,outcome
0,m_91,p_103,جریان بازی,پای راست,13.47,-11.22,1,0,متوسط,70,9,گُل
1,m_17,p_16,جریان بازی,پای چپ,9.48,14.22,3,0,متوسط,55,4,مهار توسط دروازه بان
2,m_111,p_88,ضربه آزاد مستقیم,پای چپ,29.43,-1.25,6,2,کم,86,31,مهار توسط دروازه بان
3,m_142,p_87,جریان بازی,پای راست,26.93,1.00,4,1,متوسط,77,2,موقعیت از دست رفته
4,m_117,p_9,جریان بازی,پای راست,10.72,5.24,2,0,متوسط,76,46,گُل
...,...,...,...,...,...,...,...,...,...,...,...,...
8920,m_57,p_115,جریان بازی,سر,6.48,3.99,3,0,زیاد,69,50,موقعیت از دست رفته
8921,m_59,p_76,جریان بازی,پای راست,21.45,-8.73,4,1,متوسط,15,53,برخورد به دفاع
8922,m_55,p_150,جریان بازی,پای چپ,11.97,3.24,3,0,متوسط,84,34,موقعیت از دست رفته
8923,m_33,p_130,جریان بازی,پای راست,6.48,-6.98,1,0,زیاد,4,39,موقعیت از دست رفته


In [ ]:
positive_outcomes = ["گُل", "گُل به خودی"]
df["label"] = df["outcome"].isin(positive_outcomes).astype(int)
df.drop(columns=['outcome'], inplace=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['matchId', 'playerId', 'playType', 'bodyPart', 'interferenceOnShooter', 'label'], axis=1),
    df['label'],
    random_state=313,
    stratify=df['label']
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
print(f'performance of model is {roc_auc_score(y_test, y_pred_proba)}')

performance of model is 0.7178387445470346


In [ ]:
df.drop(columns=['matchId'], inplace=True)
df.drop(columns=['playerId'], inplace=True)

In [ ]:
df['bodyPart'] = df['bodyPart'].replace({'پای چپ': 'پا', 'پای راست': 'پا'})

In [ ]:
df['distance'] = np.sqrt(df['x'] ** 2 + df['y'] ** 2)

denom = df['x'] ** 2 + df['y'] ** 2 - (7.32 / 2) ** 2
theta = 7.32 * df['x'] / denom

angle_rad = np.arctan(theta)
angle_rad = np.where(angle_rad < 0, angle_rad + np.pi, angle_rad)
df['angle'] = np.degrees(angle_rad)

df.drop(columns=['x', 'y'], inplace=True)

In [12]:
df.isna().sum()

playType                  0
bodyPart                  0
interveningOpponents      0
interveningTeammates      0
interferenceOnShooter    34
minute                    0
second                    0
label                     0
distance                  0
angle                     0
dtype: int64

In [ ]:
cond0 = df['interveningOpponents'] == 0
cond1 = df['interveningOpponents'] == 1

mask_na = df['interferenceOnShooter'].isna()

df.loc[mask_na & cond0, 'interferenceOnShooter'] = 'کم'
df.loc[mask_na & cond1, 'interferenceOnShooter'] = 'متوسط'

df.loc[mask_na & ~(cond0 | cond1), 'interferenceOnShooter'] = 'زیاد'

In [ ]:
play_dummies = pd.get_dummies(df['playType'], prefix='playType')
body_dummies = pd.get_dummies(df['bodyPart'], prefix='bodyPart')
int_dummies = pd.get_dummies(df['interferenceOnShooter'], prefix='interferenceOnShooter')


df = pd.concat([df, play_dummies, body_dummies, int_dummies], axis=1)

df.drop(columns=['playType', 'bodyPart', 'interferenceOnShooter'], inplace=True)

added_cols = list(play_dummies.columns) + list(body_dummies.columns) + list(int_dummies.columns)

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

continuous_cols = ['distance', 'angle']
discrete_mask = [col not in continuous_cols for col in X.columns]

mi = mutual_info_classif(X, y, discrete_features=discrete_mask, random_state=1401)

feature_importance = pd.DataFrame({'fi': mi}, index=X.columns)
feature_importance.sort_values('fi', ascending=False, inplace=True)

In [ ]:
model = LogisticRegression(max_iter=1000)
cols_to_train = feature_importance[feature_importance.fi >= feature_importance.fi.quantile(.5)].index
x = df[cols_to_train]
y = df.label
model.fit(x,y)
x_train,x_test , y_train,y_test = train_test_split(x,y, random_state=313, test_size=.3, stratify=y)
y_pred_proba = model.predict_proba(x_test)[:, 1]
print(f'performance of model is {roc_auc_score(y_test, y_pred_proba)}')